# Module 2: LLM Gateway (LiteLLM Proxy) — End-to-End Walkthrough

This notebook walks through the LLM Gateway features after deployment.

**Prerequisites:**
- Stack `workshop-llm-gateway-stack` deployed (Step 2)
- Virtual keys created (Step 3)
- Environment variables set: `LLM_GATEWAY_URL`, `LLM_GATEWAY_API_KEY`, `LLM_GATEWAY_ADMIN_KEY`

## 0. Setup

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

sys.path.insert(0, ".")

from llm_gateway_client import LLMGatewayClient

# Load state saved by notebooks/step-2-deploy.ipynb / step-3-virtual-keys.ipynb.
# Environment variables take precedence if set.
state = {}
state_path = Path("notebooks/.state.json")
if state_path.exists():
    state = json.loads(state_path.read_text())

PROXY_URL = os.environ.get("LLM_GATEWAY_URL") or state.get("LLM_GATEWAY_URL")
API_KEY = os.environ.get("LLM_GATEWAY_API_KEY") or state.get("API_KEY", "")
ADMIN_KEY = os.environ.get("LLM_GATEWAY_ADMIN_KEY") or state.get("ADMIN_KEY", "")

if not PROXY_URL:
    raise RuntimeError(
        "Proxy URL not found. Run notebooks/step-2-deploy.ipynb first, "
        "or export LLM_GATEWAY_URL."
    )

client = LLMGatewayClient(proxy_url=PROXY_URL, api_key=API_KEY)
admin = LLMGatewayClient(proxy_url=PROXY_URL, api_key=ADMIN_KEY)

print(f"Proxy URL:  {PROXY_URL}")
print(f"API Key:    {'*' * 8 + API_KEY[-4:] if API_KEY else '(none)'}")
print(f"Admin Key:  {'*' * 8 + ADMIN_KEY[-4:] if ADMIN_KEY else '(none)'}")


## 1. Health Check

In [ ]:
health = client.health_check()
print(json.dumps(health, indent=2))

## 2. List Available Models

In [ ]:
models = client.list_models()
print(f"Available models ({len(models.data)}):")
for m in models.data:
    print(f"  - {m.id}")

## 3. Chat Completion

In [ ]:
response = client.chat(
    prompt="What are three benefits of using an LLM Gateway in enterprise AI?",
    model="claude-sonnet",
    max_tokens=300,
)
print(response)

## 4. OpenAI SDK Compatibility

The gateway is OpenAI-compatible — the standard `openai` package works by changing the base URL.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=API_KEY,
    base_url=PROXY_URL
)

response = openai_client.chat.completions.create(
    model="claude-sonnet",
    messages=[{"role": "user", "content": "Hello from the OpenAI SDK through LiteLLM Proxy!"}],
    max_tokens=100,
)
print(response.choices[0].message.content)

## 5. Multi-Model Routing

Switch between Bedrock models by changing the `model` parameter — no code changes.

In [ ]:
models_to_test = ["claude-sonnet", "claude-haiku", "nova-pro", "nova-lite"]

for model_id in models_to_test:
    try:
        response = client.chat(
            prompt="In one sentence, what makes you unique?",
            model=model_id,
            max_tokens=100,
        )
        print(f"[{model_id}]")
        print(f"  {response}\n")
    except Exception as e:
        print(f"[{model_id}] Failed: {e}\n")

## 6. Caching Demonstration

Send the same request twice. The second should be faster (cache hit).

In [ ]:
prompt = "What is 2 + 2? Reply with just the number."

# First request
start = time.time()
r1 = client.chat(prompt=prompt, model="claude-sonnet", max_tokens=10, temperature=0)
t1 = time.time() - start
print(f"Request 1: '{r1}' ({t1:.2f}s)")

# Second request (should be cached)
start = time.time()
r2 = client.chat(prompt=prompt, model="claude-sonnet", max_tokens=10, temperature=0)
t2 = time.time() - start
print(f"Request 2: '{r2}' ({t2:.2f}s)")

if t2 < t1 * 0.5:
    print(f"\nCache hit! {t1/t2:.1f}x faster.")
else:
    print("\nSimilar timing — caching may need a moment to warm up.")

## 7. Virtual Key Management

Create a new virtual key with a budget and check its spend.

In [ ]:
# Create a new key (requires admin key)
new_key = admin.create_key(
    models=["claude-sonnet", "nova-lite"],
    max_budget=2.0,
)
print(f"New key: (value not printed)")
print(f"Budget:  ${new_key.max_budget}")

In [ ]:
# Check key info
key_info = admin.get_key_info(new_key.key)
print(json.dumps(key_info, indent=2, default=str))

## 8. Spend Tracking

In [ ]:
logs = admin.get_spend_logs()
if logs:
    total_spend = sum(log.get("spend", 0) for log in logs)
    print(f"Total requests: {len(logs)}")
    print(f"Total spend:    ${total_spend:.6f}")
    print()
    for log in logs[:5]:
        print(f"  model={log.get('model', '?'):20s}  "
              f"tokens={log.get('total_tokens', 0):5d}  "
              f"${log.get('spend', 0):.6f}")
else:
    print("No spend logs yet. Send a few requests first.")

## 9. Strands Agent Integration

This is the critical integration point for Module 5. The agent routes all LLM calls through the gateway.

In [ ]:
try:
    from strands import Agent
    from strands.models.litellm import LiteLLMModel

    # The "openai/" prefix tells litellm to use the OpenAI-compatible
    # protocol, which the LiteLLM Proxy speaks natively.
    model = LiteLLMModel(
        model_id="openai/claude-sonnet",
        params={
            "api_base": PROXY_URL,
            "api_key": API_KEY,
        },
    )

    agent = Agent(model=model)
    result = agent("What is 15 * 23? Think step by step and give the answer.")
    print(f"Agent response: {result}")
    print()
    print("Strands Agent -> LiteLLM Proxy -> Bedrock is working!")
except ImportError:
    print("strands-agents not installed. Install the pinned workshop version with:")
    print("  pip install 'strands-agents[litellm]==0.1.5'")
except Exception as e:
    print(f"Agent test failed: {e}")

## 10. Cleanup

In the Workshop Studio workshop, the LLM Gateway stack (`workshop-llm-gateway-stack`) is pre-provisioned and will be cleaned up automatically at event end. Do **not** delete it.

For self-service deployments in your own account:

```bash
aws cloudformation delete-stack --stack-name workshop-llm-gateway-stack
```
